**3. ML training pipeline.**

We will take following features to train model
1. Raw sensor data(tenperature, vibration, humidity, pressure, energy_consumption) 
2. Engineered features (z score of all sensors data, changing rate, rolling features)

We will build a model that learns, Given sensor behavior → is the machine anomalous or not?
 Target -> anomaly_flag (renamed as label),      Input -> df_model

In [0]:
from pyspark.sql.functions import col

df_feat = spark.table("feature_data")


#ml_data = df_feat.withColumnRenamed("anomaly_flag", "label")
ml_data = df_feat.withColumn(
    "label",
    col("anomaly_flag")
)

#Let's take a look at the feature set
#“A full predictive maintenance feature set combining raw sensors + deviation + anomaly score + time trends”
feature_cols = [

    # =========================
    # RAW SENSOR DATA
    # =========================
    "temperature",
    "vibration",
    "humidity",
    "pressure",
    "energy_consumption",

    # =========================
    # DEVIATION FEATURES
    # =========================
    "temp_dev",
    "vib_dev",
    "energy_dev",
    "pressure_dev",

    # =========================
    # Z-SCORE FEATURES
    # =========================
    "temp_z",
    "vib_z",
    "energy_z",
    "pressure_z",

    # =========================
    # TIME BEHAVIOR FEATURES
    # =========================
    "temperature_change",
    "temperature_change_abs",
    "temperature_roll_mean",
    "temperature_roll_std",

    "vibration_change",
    "vibration_change_abs",
    "vibration_roll_mean",
    "vibration_roll_std",

    "humidity_change",
    "humidity_change_abs",
    "humidity_roll_mean",
    "humidity_roll_std",

    "pressure_change",
    "pressure_change_abs",
    "pressure_roll_mean",
    "pressure_roll_std",

    "energy_consumption_change",
    "energy_consumption_change_abs",
    "energy_consumption_roll_mean",
    "energy_consumption_roll_std"
]

df_model = ml_data.select(
    "timestamp",
    "machine_id",
    "label",
    *feature_cols
)
df_model.show(5)
#machine_status, anomaly_flag these are not included in this model for label leakage issue


We here do not using random splitting but time-based splitting because out problem is to predict machine failure in the future

In [0]:
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml import Pipeline
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.sql.functions import col, percent_rank
from pyspark.sql.window import Window


# =========================
# 2. VECTOR ASSEMBLER
# =========================
assembler = VectorAssembler(
    inputCols=feature_cols,
    outputCol="features"
)

# =========================
# 3. Train/Test split
# =========================
time_window = Window.orderBy("timestamp")

df_time = df_model.withColumn(
    "time_pct",
    percent_rank().over(time_window)
)

train_data = df_time.filter(col("time_pct") <= 0.8)

test_data = df_time.filter(col("time_pct") > 0.8)


# =========================
# 4. Handle class imbalance
# =========================
from pyspark.sql.functions import when
df_model = df_model.withColumn(
    "classWeight",
    when(col("label") == 1, 3.0).otherwise(1.0)
)

# =========================
# 5. Model (Random Forest)
# =========================
rf = RandomForestClassifier(
    labelCol="label",
    featuresCol="features",
    numTrees=100,
    maxDepth=10
)

# =========================
# 5. Pipeline
# =========================
pipeline = Pipeline(stages=[assembler, rf])

# =========================
# 6. Train model (ANOMALY PREDICTION MODEL)
# =========================
model = pipeline.fit(train_data)

# =========================
# 6. Predictions
# =========================
predictions = model.transform(test_data)
predictions.select(
    "timestamp",
    "machine_id",
    "label",
    "prediction",
    "probability"
).show(20, False)

# =========================
# 6. Evaluation of model (AUC)
# =========================
evaluator = BinaryClassificationEvaluator(
    labelCol="label",
    rawPredictionCol="rawPrediction"
)
auc = evaluator.evaluate(predictions)

print("Model AUC:", auc)

predictions.groupBy(
    "label",
    "prediction"
).count().show()

### ## Result Interpretation
The trained model demonstrates excellent predictive performance with an AUC of 0.9991, indicating near-perfect separation between normal and anomalous machine states.

The confusion matrix shows:
- High True Positive rate (1639 correctly detected anomalies)
- Very low False Positive rate (42 false alarms)
- Low False Negative rate (172 missed anomalies)

This indicates that the model is highly effective for industrial predictive maintenance, with strong anomaly detection capability and minimal unnecessary alerts.

Overall, the system is suitable for real-time machine health monitoring in an Industry 4.0 environment.

In [0]:
predictions.select(
    "machine_id",
    "label",
    "prediction"
).show(10)

from pyspark.sql.functions import col, avg, count

result = predictions.withColumn(
    "correct",
    (col("label") == col("prediction")).cast("int")
)
machine_accuracy = result.groupBy("machine_id").agg(
    avg("correct").alias("accuracy"),
    count("*").alias("total_samples")
)

machine_accuracy.orderBy(col("accuracy").desc()).show(5)
machine_accuracy.orderBy(col("accuracy").asc()).show(5)

**Machine-level accuracy analysis**

The machine-level evaluation shows consistently high prediction accuracy across all machines, ranging from 0.976 to 1.0.

Best-performing machines achieve near-perfect accuracy (~100%), indicating strong separability between normal and anomalous behavior.

Lower-performing machines still maintain accuracy above 97%, suggesting robust model generalization across different operational patterns.

Overall, the system demonstrates high reliability for predictive maintenance and is suitable for industrial deployment scenarios where early anomaly detection is critical.

In [0]:
predictions.write.mode("overwrite").saveAsTable("predictions")
machine_accuracy.write.mode("overwrite").saveAsTable("machine_accuracy")